In [2]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../..')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Bucket('aws'), Bucket('azure'), Bucket('google')]


In [3]:
object_name = "Compute_Engine/page_1.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


Success. Page loaded and converted into dataframe
Array size: Rows =  5000  and Columns =  2


In [28]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
# df_flat.head(3)
df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

,skuId,category.serviceDisplayName,category.resourceFamily,category.usageType,category.resourceGroup
0,0001-B904-8A40,Compute Engine,Compute,OnDemand,CPU
1,0001-FC8F-A9AF,Compute Engine,Compute,Preemptible,CPU
2,0006-C9C8-BB6F,Compute Engine,Compute,Commit1Yr,CPU
3,0007-4724-5A32,Compute Engine,Compute,OnDemand,CPU
4,0007-9388-EF75,Compute Engine,Compute,OnDemand,RAM


In [39]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions')

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[southamerica-west1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,[europe-west9],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,[us-west8],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8


In [ ]:
#We apply the same proccedure as the cell above. This time we open the list serviceRegions
df_flat2[['skuId', 'serviceRegions']].head()

df_flat3 = df_flat2.explode('serviceRegions')

df_flat3[['skuId', 'serviceRegions', 'geoTaxonomy.regions']].head()


,skuId,serviceRegions,geoTaxonomy.regions
0,0001-B904-8A40,southamerica-west1,southamerica-west1
1,0001-FC8F-A9AF,europe-west9,europe-west9
2,0006-C9C8-BB6F,us-west8,us-west8
3,0007-4724-5A32,europe-north1,europe-north1
4,0007-9388-EF75,northamerica-northeast2,northamerica-northeast2
